# Landslide Hazard Analysis: Map Overview

This notebook reads and visualizes the three landslide probability maps in:
- `dphil_paper_3/inputs/landslides`

Maps loaded:
- `landslide_probability_Baseline.tif`
- `landslide_probability_Deforestation.tif`
- `landslide_probability_Reafforestation.tif`


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import BoundaryNorm, ListedColormap, TwoSlopeNorm
from rasterio.warp import Resampling, calculate_default_transform, reproject


In [ ]:
def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start] + list(start.parents):
        if (p / 'dphil_papers').exists():
            return p
    raise FileNotFoundError(f'Could not find project root from {start}')

ROOT = find_project_root(Path.cwd())
landslides_dir = ROOT / 'dphil_papers/dphil_paper_3/inputs/landslides'

raster_paths = [
    landslides_dir / 'landslide_probability_Baseline.tif',
    landslides_dir / 'landslide_probability_Deforestation.tif',
    landslides_dir / 'landslide_probability_Reafforestation.tif',
]

for p in raster_paths:
    print(p.name, 'exists ->', p.exists())


In [ ]:
# Read rasters, convert to Jamaica projected CRS, and summarize metadata

TARGET_CRS = rasterio.crs.CRS.from_epsg(3448)
NODATA_FILL = -9999.0

def read_reprojected_raster(path, target_crs, dst_transform=None, dst_width=None, dst_height=None, resampling=Resampling.bilinear):
    with rasterio.open(path) as src:
        src_arr = src.read(1).astype(np.float32)
        src_nodata = src.nodata
        if src_nodata is None or not np.isfinite(src_nodata):
            src_nodata = NODATA_FILL

        if dst_transform is None or dst_width is None or dst_height is None:
            dst_transform, dst_width, dst_height = calculate_default_transform(
                src.crs,
                target_crs,
                src.width,
                src.height,
                *src.bounds,
            )

        dst_arr = np.full((dst_height, dst_width), NODATA_FILL, dtype=np.float32)
        reproject(
            source=src_arr,
            destination=dst_arr,
            src_transform=src.transform,
            src_crs=src.crs,
            src_nodata=src_nodata,
            dst_transform=dst_transform,
            dst_crs=target_crs,
            dst_nodata=NODATA_FILL,
            resampling=resampling,
        )

    b = rasterio.transform.array_bounds(dst_height, dst_width, dst_transform)
    return {
        'array': dst_arr,
        'transform': dst_transform,
        'bounds': rasterio.coords.BoundingBox(left=b[0], bottom=b[1], right=b[2], top=b[3]),
        'crs': target_crs,
        'shape': dst_arr.shape,
        'nodata': NODATA_FILL,
    }

rasters = {}
meta_rows = []
ref_transform = None
ref_shape = None

for p in raster_paths:
    rr = read_reprojected_raster(
        p,
        target_crs=TARGET_CRS,
        dst_transform=ref_transform,
        dst_width=None if ref_shape is None else ref_shape[1],
        dst_height=None if ref_shape is None else ref_shape[0],
        resampling=Resampling.bilinear,
    )
    if ref_transform is None:
        ref_transform = rr['transform']
        ref_shape = rr['shape']

    arr = rr['array']
    nodata = rr['nodata']
    valid = np.isfinite(arr) & (arr != nodata)
    vals = arr[valid]

    rasters[p.stem] = {
        **rr,
        'valid_mask': valid,
    }

    meta_rows.append({
        'map': p.stem,
        'crs': str(rr['crs']),
        'shape': str(rr['shape']),
        'nodata': nodata,
        'min': float(np.min(vals)) if vals.size else np.nan,
        'max': float(np.max(vals)) if vals.size else np.nan,
        'mean': float(np.mean(vals)) if vals.size else np.nan,
        'std': float(np.std(vals)) if vals.size else np.nan,
    })

meta_df = pd.DataFrame(meta_rows).round(4)
display(meta_df)


In [ ]:
# Check if all rasters are aligned on the same grid
keys = list(rasters.keys())
base = rasters[keys[0]]

aligned = True
for k in keys[1:]:
    r = rasters[k]
    same = (r['crs'] == base['crs']) and (r['shape'] == base['shape']) and (r['transform'] == base['transform'])
    print(k, 'aligned with', keys[0], '->', same)
    aligned = aligned and same

print('All aligned:', aligned)


In [ ]:
# Plot all 3 maps side-by-side with a shared color scale
arrays = [rasters[k]['array'] for k in keys]

# global valid min/max for consistent color scaling
global_min = np.inf
global_max = -np.inf
for a, k in zip(arrays, keys):
    nodata = rasters[k]['nodata']
    valid = np.isfinite(a)
    if nodata is not None and np.isfinite(nodata):
        valid &= a != nodata
    if np.any(valid):
        global_min = min(global_min, float(np.min(a[valid])))
        global_max = max(global_max, float(np.max(a[valid])))

fig, axes = plt.subplots(1, 3, figsize=(16, 5.2), constrained_layout=True)

for ax, k in zip(axes, keys):
    r = rasters[k]
    b = r['bounds']
    im = ax.imshow(
        r['array'],
        extent=[b.left, b.right, b.bottom, b.top],
        origin='upper',
        cmap='YlOrRd',
        vmin=global_min,
        vmax=global_max,
        interpolation='nearest',
    )
    ax.set_title(k.replace('landslide_probability_', ''))
    ax.set_xlabel('Easting')
    ax.set_ylabel('Northing')
    ax.set_aspect('equal')

cbar = fig.colorbar(im, ax=axes, shrink=0.9, pad=0.02)
cbar.set_label('Landslide probability')

plt.show()


In [ ]:
# Distribution comparison
fig, ax = plt.subplots(figsize=(8.5, 4.8), constrained_layout=True)

for k in keys:
    a = rasters[k]['array']
    nodata = rasters[k]['nodata']
    valid = np.isfinite(a)
    if nodata is not None and np.isfinite(nodata):
        valid &= a != nodata
    vals = a[valid]
    ax.hist(vals, bins=60, density=True, alpha=0.35, label=k.replace('landslide_probability_', ''))

ax.set_title('Landslide Probability Distributions')
ax.set_xlabel('Probability')
ax.set_ylabel('Density')
ax.legend(frameon=True)
plt.show()


## Added at End: Difference Analysis (Maps Are Similar but Not Identical)

These layers can look visually similar with the same absolute color range. This section quantifies and maps **differences** directly.


In [ ]:
# Pairwise difference statistics and delta maps

name_map = {
    'landslide_probability_Baseline': 'Baseline',
    'landslide_probability_Deforestation': 'Deforestation',
    'landslide_probability_Reafforestation': 'Reafforestation',
}

# Ensure predictable ordering
keys_named = []
for k in ['landslide_probability_Baseline', 'landslide_probability_Deforestation', 'landslide_probability_Reafforestation']:
    if k in rasters:
        keys_named.append(k)

if len(keys_named) < 3:
    raise ValueError('Expected all 3 rasters in `rasters`.')

A = rasters['landslide_probability_Baseline']['array']
B = rasters['landslide_probability_Deforestation']['array']
C = rasters['landslide_probability_Reafforestation']['array']

nod_A = rasters['landslide_probability_Baseline']['nodata']
nod_B = rasters['landslide_probability_Deforestation']['nodata']
nod_C = rasters['landslide_probability_Reafforestation']['nodata']

valid_A = np.isfinite(A) & ((A != nod_A) if (nod_A is not None and np.isfinite(nod_A)) else True)
valid_B = np.isfinite(B) & ((B != nod_B) if (nod_B is not None and np.isfinite(nod_B)) else True)
valid_C = np.isfinite(C) & ((C != nod_C) if (nod_C is not None and np.isfinite(nod_C)) else True)

pair_defs = [
    ('Deforestation - Baseline', B, A, valid_B & valid_A),
    ('Reafforestation - Baseline', C, A, valid_C & valid_A),
    ('Deforestation - Reafforestation', B, C, valid_B & valid_C),
]

stats_rows = []
diff_arrays = {}
for label, X, Y, m in pair_defs:
    d = np.full(X.shape, np.nan, dtype='float32')
    d[m] = (X[m] - Y[m]).astype('float32')
    diff_arrays[label] = d

    vals = d[m]
    stats_rows.append({
        'comparison': label,
        'n_valid': int(vals.size),
        'mean_diff': float(np.mean(vals)),
        'std_diff': float(np.std(vals)),
        'min_diff': float(np.min(vals)),
        'max_diff': float(np.max(vals)),
        'pct_exact_equal': float(100.0 * np.mean(vals == 0.0)),
        'pct_absdiff_gt_0.001': float(100.0 * np.mean(np.abs(vals) > 0.001)),
        'pct_absdiff_gt_0.01': float(100.0 * np.mean(np.abs(vals) > 0.01)),
    })

stats_df = pd.DataFrame(stats_rows).round(4)
display(stats_df)

# Plot delta maps with robust symmetric scaling around zero
bnds = rasters['landslide_probability_Baseline']['bounds']
fig, axes = plt.subplots(1, 3, figsize=(17, 5.4), constrained_layout=True)

for ax, (label, d) in zip(axes, diff_arrays.items()):
    vals = d[np.isfinite(d)]
    if vals.size == 0:
        lim = 1e-6
    else:
        # Robust limit to make subtle differences visible
        lim = float(np.percentile(np.abs(vals), 99))
        lim = max(lim, 1e-5)

    norm = TwoSlopeNorm(vmin=-lim, vcenter=0.0, vmax=lim)
    im = ax.imshow(
        d,
        extent=[bnds.left, bnds.right, bnds.bottom, bnds.top],
        origin='upper',
        cmap='RdBu_r',
        norm=norm,
        interpolation='nearest',
    )
    ax.set_title(label)
    ax.set_xlabel('Easting')
    ax.set_ylabel('Northing')
    ax.set_aspect('equal')
    cb = fig.colorbar(im, ax=ax, shrink=0.9, pad=0.02)
    cb.set_label('Delta probability')

plt.show()


## Added at End: Areas at Risk (Probability >= 0.5)

This section treats pixels with landslide probability `>= 0.5` as at-risk and summarizes the extent for each scenario.


In [ ]:
# Binary risk analysis: probability >= 0.5
RISK_THRESHOLD = 0.5

risk_rows = []
risk_masks = {}

for k in keys:
    r = rasters[k]
    a = r['array']
    nodata = r['nodata']

    valid = np.isfinite(a)
    if nodata is not None and np.isfinite(nodata):
        valid &= a != nodata

    risk = valid & (a >= RISK_THRESHOLD)
    risk_masks[k] = risk

    n_valid = int(np.sum(valid))
    n_risk = int(np.sum(risk))
    pct_risk = 100.0 * n_risk / max(n_valid, 1)

    # Area is meaningful because rasters have been projected to Jamaica CRS (EPSG:3448)
    if r['crs'] is not None and getattr(r['crs'], 'is_projected', False):
        px_area = abs(r['transform'].a * r['transform'].e)
        area_risk_km2 = (n_risk * px_area) / 1_000_000.0
    else:
        area_risk_km2 = np.nan

    risk_rows.append({
        'scenario': k.replace('landslide_probability_', ''),
        'threshold': RISK_THRESHOLD,
        'n_valid_pixels': n_valid,
        'n_at_risk_pixels': n_risk,
        'pct_at_risk_pixels': pct_risk,
        'at_risk_area_km2': area_risk_km2,
    })

risk_df = pd.DataFrame(risk_rows)
risk_df[['pct_at_risk_pixels', 'at_risk_area_km2']] = risk_df[['pct_at_risk_pixels', 'at_risk_area_km2']].round(3)
risk_df['at_risk_area_km2'] = risk_df['at_risk_area_km2'].where(~risk_df['at_risk_area_km2'].isna(), other=np.nan)
display(risk_df)

# Plot binary risk maps
fig, axes = plt.subplots(1, 3, figsize=(16, 5.2), constrained_layout=True)
for ax, k in zip(axes, keys):
    r = rasters[k]
    b = r['bounds']
    risk = risk_masks[k]
    show = np.where(risk, 1.0, 0.0)

    ax.imshow(
        show,
        extent=[b.left, b.right, b.bottom, b.top],
        origin='upper',
        cmap=ListedColormap(['#d9d9d9', '#d7301f']),
        vmin=0,
        vmax=1,
        interpolation='nearest',
    )
    ax.set_title(k.replace('landslide_probability_', ''))
    ax.set_xlabel('Easting')
    ax.set_ylabel('Northing')
    ax.set_aspect('equal')

legend_handles = [
    mpatches.Patch(facecolor='#d7301f', edgecolor='none', label=f'At risk (>= {RISK_THRESHOLD})'),
    mpatches.Patch(facecolor='#d9d9d9', edgecolor='none', label='Below threshold'),
]
fig.legend(handles=legend_handles, loc='lower center', ncol=2, frameon=True)
plt.show()

# Quick comparison chart
fig, ax = plt.subplots(figsize=(7.2, 4.2), constrained_layout=True)
ax.bar(risk_df['scenario'], risk_df['pct_at_risk_pixels'], color=['#756bb1', '#d7301f', '#1a9850'])
ax.set_ylabel('% pixels at risk (>= 0.5)')
ax.set_title('At-Risk Share by Scenario')
for i, v in enumerate(risk_df['pct_at_risk_pixels']):
    ax.text(i, v + 0.4, f'{v:.2f}%', ha='center', va='bottom', fontsize=9)
plt.show()


## Added at End: Where At-Risk Pixels Change Across Scenarios (`>=0.5`)

This section focuses only on pixels that are at risk in **any** scenario (`probability >= 0.5`) and maps which scenario combination each pixel belongs to.


In [ ]:
# Visual change map focused on union of at-risk pixels (>=0.5 in any scenario)

# Rebuild risk masks robustly
RISK_THRESHOLD = 0.5
A = rasters['landslide_probability_Baseline']['array']
B = rasters['landslide_probability_Deforestation']['array']
C = rasters['landslide_probability_Reafforestation']['array']

nod_A = rasters['landslide_probability_Baseline']['nodata']
nod_B = rasters['landslide_probability_Deforestation']['nodata']
nod_C = rasters['landslide_probability_Reafforestation']['nodata']

valid_A = np.isfinite(A) & ((A != nod_A) if (nod_A is not None and np.isfinite(nod_A)) else True)
valid_B = np.isfinite(B) & ((B != nod_B) if (nod_B is not None and np.isfinite(nod_B)) else True)
valid_C = np.isfinite(C) & ((C != nod_C) if (nod_C is not None and np.isfinite(nod_C)) else True)
valid_all = valid_A & valid_B & valid_C

risk_A = valid_all & (A >= RISK_THRESHOLD)
risk_B = valid_all & (B >= RISK_THRESHOLD)
risk_C = valid_all & (C >= RISK_THRESHOLD)

union_risk = risk_A | risk_B | risk_C

# Category encoding over union_risk only:
# 1: Baseline only
# 2: Deforestation only
# 3: Reafforestation only
# 4: Baseline + Deforestation
# 5: Baseline + Reafforestation
# 6: Deforestation + Reafforestation
# 7: All three
cat = np.full(A.shape, np.nan, dtype='float32')

cat[ union_risk &  risk_A & ~risk_B & ~risk_C ] = 1
cat[ union_risk & ~risk_A &  risk_B & ~risk_C ] = 2
cat[ union_risk & ~risk_A & ~risk_B &  risk_C ] = 3
cat[ union_risk &  risk_A &  risk_B & ~risk_C ] = 4
cat[ union_risk &  risk_A & ~risk_B &  risk_C ] = 5
cat[ union_risk & ~risk_A &  risk_B &  risk_C ] = 6
cat[ union_risk &  risk_A &  risk_B &  risk_C ] = 7

label_map = {
    1: 'Baseline only',
    2: 'Deforestation only',
    3: 'Reafforestation only',
    4: 'Baseline + Deforestation',
    5: 'Baseline + Reafforestation',
    6: 'Deforestation + Reafforestation',
    7: 'All three scenarios',
}

# Summary table
rows = []
n_union = int(np.sum(union_risk))
for k in range(1, 8):
    n = int(np.nansum(cat == float(k)))
    rows.append({'category': label_map[k], 'n_pixels': n, 'pct_of_union_at_risk': 100.0 * n / max(n_union, 1)})

combo_df = pd.DataFrame(rows)
combo_df['pct_of_union_at_risk'] = combo_df['pct_of_union_at_risk'].round(3)
display(combo_df)

# A compact change summary
stable_all_three = int(np.nansum(cat == 7.0))
changed_across_scenarios = n_union - stable_all_three
print(f'Union at-risk pixels (>= {RISK_THRESHOLD} in any scenario): {n_union:,}')
print(f'At-risk in all three scenarios (stable): {stable_all_three:,} ({100*stable_all_three/max(n_union,1):.2f}%)')
print(f'At-risk but scenario-dependent (changes): {changed_across_scenarios:,} ({100*changed_across_scenarios/max(n_union,1):.2f}%)')

# Plot categorical map
b = rasters['landslide_probability_Baseline']['bounds']

colors = [
    '#8c510a',  # 1 baseline only
    '#d7301f',  # 2 deforestation only
    '#1a9850',  # 3 reaff only
    '#fc8d59',  # 4 base+defor
    '#66c2a5',  # 5 base+reaff
    '#80b1d3',  # 6 defor+reaff
    '#6a3d9a',  # 7 all three
]

cmap = ListedColormap(colors)
cmap.set_bad(color='white', alpha=1.0)
norm = BoundaryNorm(np.arange(0.5, 8.5, 1), cmap.N)

fig, ax = plt.subplots(figsize=(10.5, 7.0), constrained_layout=True)
ax.imshow(
    cat,
    extent=[b.left, b.right, b.bottom, b.top],
    origin='upper',
    cmap=cmap,
    norm=norm,
    interpolation='nearest',
)

ax.set_title('At-Risk Pixels (>=0.5) by Scenario Combination')
ax.set_xlabel('Easting')
ax.set_ylabel('Northing')
ax.set_aspect('equal')

handles = [mpatches.Patch(facecolor=colors[i-1], edgecolor='none', label=label_map[i]) for i in range(1, 8)]
ax.legend(handles=handles, loc='upper right', frameon=True, fontsize=8)

plt.show()


## Added at End: New Probability-Change Maps

Read and visualize the newly added change rasters:
- `probability_change_Deforestation.tif`
- `probability_change_Reafforestation.tif`


In [ ]:
# Read new probability-change maps and visualize

change_paths = [
    landslides_dir / 'probability_change_Deforestation.tif',
    landslides_dir / 'probability_change_Reafforestation.tif',
]

for p in change_paths:
    print(p.name, 'exists ->', p.exists())

change_rasters = {}
rows = []
base = rasters['landslide_probability_Baseline']

for p in change_paths:
    rr = read_reprojected_raster(
        p,
        target_crs=TARGET_CRS,
        dst_transform=base['transform'],
        dst_width=base['shape'][1],
        dst_height=base['shape'][0],
        resampling=Resampling.bilinear,
    )
    arr = rr['array']
    nodata = rr['nodata']
    valid = np.isfinite(arr) & (arr != nodata)
    vals = arr[valid]

    change_rasters[p.stem] = {
        **rr,
        'valid_mask': valid,
    }

    rows.append({
        'map': p.stem,
        'crs': str(rr['crs']),
        'shape': str(rr['shape']),
        'nodata': nodata,
        'min': float(np.min(vals)) if vals.size else np.nan,
        'max': float(np.max(vals)) if vals.size else np.nan,
        'mean': float(np.mean(vals)) if vals.size else np.nan,
        'std': float(np.std(vals)) if vals.size else np.nan,
        'p01': float(np.percentile(vals, 1)) if vals.size else np.nan,
        'p99': float(np.percentile(vals, 99)) if vals.size else np.nan,
    })

change_df = pd.DataFrame(rows).round(6)
display(change_df)

# Check alignment with baseline probability raster
for k, r in change_rasters.items():
    same = (r['crs'] == base['crs']) and (r['shape'] == base['shape']) and (r['transform'] == base['transform'])
    print(k, 'aligned with baseline grid ->', same)

# Plot both change maps with symmetric, zero-centered scaling
keys_c = list(change_rasters.keys())

all_vals = []
for k in keys_c:
    rr = change_rasters[k]
    v = rr['array'][rr['valid_mask']]
    all_vals.append(v)
all_vals = np.concatenate(all_vals) if len(all_vals) else np.array([0.0])
lim = float(np.percentile(np.abs(all_vals), 99)) if all_vals.size else 1e-6
lim = max(lim, 1e-6)

fig, axes = plt.subplots(1, 2, figsize=(12.8, 5.4), constrained_layout=True)
for ax, k in zip(axes, keys_c):
    rr = change_rasters[k]
    b = rr['bounds']
    im = ax.imshow(
        rr['array'],
        extent=[b.left, b.right, b.bottom, b.top],
        origin='upper',
        cmap='RdBu_r',
        norm=TwoSlopeNorm(vmin=-lim, vcenter=0.0, vmax=lim),
        interpolation='nearest',
    )
    ax.set_title(k.replace('probability_change_', 'Change: '))
    ax.set_xlabel('Easting')
    ax.set_ylabel('Northing')
    ax.set_aspect('equal')
    cb = fig.colorbar(im, ax=ax, shrink=0.92, pad=0.02)
    cb.set_label('Probability change')

plt.show()

# Distribution comparison for change values
fig, ax = plt.subplots(figsize=(8.5, 4.6), constrained_layout=True)
for k in keys_c:
    rr = change_rasters[k]
    v = rr['array'][rr['valid_mask']]
    ax.hist(v, bins=70, density=True, alpha=0.4, label=k.replace('probability_change_', ''))
ax.axvline(0.0, color='black', linewidth=1.0, linestyle='--')
ax.set_title('Distribution of Probability-Change Values')
ax.set_xlabel('Probability change')
ax.set_ylabel('Density')
ax.legend(frameon=True)
plt.show()
